# NB 02 — Validación SUNAT
**Proyecto 5 · Automatización Contable**

**Input:** `data/processed/facturas_extraidas.json`  
**Output:** `data/processed/facturas_validadas.json`

Valida el RUC de cada factura contra la API pública de SUNAT via `apis.net.pe`.  
No detiene el pipeline — marca las anomalías para revisión posterior.

import json
import time
import requests
import os
from pathlib import Path
from datetime import datetime
from dotenv import load_dotenv

load_dotenv(dotenv_path=Path('D:/Proyecto_Gabriel/02_Agente_IA/Skill_financiero/.env'))

BASE_DIR      = Path('../')
INPUT_PATH    = BASE_DIR / 'data/processed/facturas_extraidas.json'
OUTPUT_PATH   = BASE_DIR / 'data/processed/facturas_validadas.json'

SUNAT_API_URL = 'https://api.apis.net.pe/v2/sunat/ruc'
HEADERS       = {'Referer': 'https://apis.net.pe'}
TIMEOUT_SEG   = 8
PAUSA_SEG     = 0.3

print('Setup OK')


In [1]:
import json
import time
import requests
from pathlib import Path
from datetime import datetime

BASE_DIR      = Path('../')
INPUT_PATH    = BASE_DIR / 'data/processed/facturas_extraidas.json'
OUTPUT_PATH   = BASE_DIR / 'data/processed/facturas_validadas.json'

SUNAT_API_URL = 'https://api.apis.net.pe/v2/sunat/ruc'
HEADERS       = {'Referer': 'https://apis.net.pe'}  # requerido por la API pública
TIMEOUT_SEG   = 8
PAUSA_SEG     = 0.3  # delay entre llamadas para no saturar la API gratuita

print('Setup OK')

Setup OK


## 1. Función de validación

In [2]:
def validar_ruc(ruc: str) -> dict:
    """
    Consulta el RUC en la API pública de SUNAT.
    Retorna dict con campos de validación, nunca lanza excepción.
    """
    if not ruc or len(str(ruc).strip()) not in (11,):
        return {
            'sunat_estado': None,
            'sunat_condicion': None,
            'sunat_razon_social_oficial': None,
            'sunat_tipo_contribuyente': None,
            'validacion_ok': False,
            'validacion_nota': f'RUC inválido: "{ruc}" (debe tener 11 dígitos)'
        }

    ruc = str(ruc).strip()

    for intento in range(2):  # 1 reintento ante fallo de red
        try:
            resp = requests.get(
                SUNAT_API_URL,
                params={'numero': ruc},
                headers=HEADERS,
                timeout=TIMEOUT_SEG
            )
            if resp.status_code == 200:
                data = resp.json()
                estado    = data.get('estado', '').upper()
                condicion = data.get('condicion', '').upper()
                return {
                    'sunat_estado': estado,
                    'sunat_condicion': condicion,
                    'sunat_razon_social_oficial': data.get('razonSocial'),
                    'sunat_tipo_contribuyente': data.get('tipoContribuyente'),
                    'validacion_ok': (estado == 'ACTIVO' and condicion == 'HABIDO'),
                    'validacion_nota': None
                }
            elif resp.status_code == 404:
                return {
                    'sunat_estado': 'NO ENCONTRADO',
                    'sunat_condicion': None,
                    'sunat_razon_social_oficial': None,
                    'sunat_tipo_contribuyente': None,
                    'validacion_ok': False,
                    'validacion_nota': f'RUC {ruc} no existe en SUNAT'
                }
            else:
                if intento == 0:
                    time.sleep(1)
                    continue
                return {
                    'sunat_estado': None, 'sunat_condicion': None,
                    'sunat_razon_social_oficial': None, 'sunat_tipo_contribuyente': None,
                    'validacion_ok': None,
                    'validacion_nota': f'HTTP {resp.status_code} en intento {intento+1}'
                }
        except requests.Timeout:
            if intento == 0:
                time.sleep(1)
                continue
            return {
                'sunat_estado': None, 'sunat_condicion': None,
                'sunat_razon_social_oficial': None, 'sunat_tipo_contribuyente': None,
                'validacion_ok': None,
                'validacion_nota': 'Timeout — API SUNAT no disponible'
            }
        except Exception as e:
            return {
                'sunat_estado': None, 'sunat_condicion': None,
                'sunat_razon_social_oficial': None, 'sunat_tipo_contribuyente': None,
                'validacion_ok': None,
                'validacion_nota': f'Error inesperado: {e}'
            }

## 2. Procesamiento en lote

In [3]:
with open(INPUT_PATH, encoding='utf-8') as f:
    facturas = json.load(f)

print(f'Facturas a validar: {len(facturas)}')

# Cachear RUCs únicos para no llamar dos veces al mismo
ruc_cache: dict[str, dict] = {}

for factura in facturas:
    tipo_op = factura.get('tipo_operacion', '')

    # El RUC a validar depende del tipo de operación:
    # COMPRA → validar RUC del emisor (proveedor)
    # VENTA  → validar RUC del receptor (cliente)
    if tipo_op == 'COMPRA':
        ruc = str(factura.get('ruc_emisor', '') or '').strip()
    else:
        ruc = str(factura.get('ruc_receptor', '') or '').strip()

    if ruc not in ruc_cache:
        resultado = validar_ruc(ruc)
        ruc_cache[ruc] = resultado
        estado_txt = resultado.get('validacion_nota') or f"{resultado.get('sunat_estado')} / {resultado.get('sunat_condicion')}"
        ok_icon = '✅' if resultado['validacion_ok'] is True else ('⚠️' if resultado['validacion_ok'] is None else '❌')
        print(f'  {ok_icon} RUC {ruc} → {estado_txt}')
        time.sleep(PAUSA_SEG)

    # Agregar campos de validación a la factura
    factura.update(ruc_cache[ruc])

    # Marcar para revisión si la validación falló o fue indeterminada
    if factura.get('validacion_ok') is not True:
        factura['requiere_revision'] = True

print(f'\nRUCs únicos consultados: {len(ruc_cache)}')

Facturas a validar: 3


  ⚠️ RUC 20613742825 → HTTP 401 en intento 2



RUCs únicos consultados: 1


## 3. Guardar y revisar resultados

In [4]:
with open(OUTPUT_PATH, 'w', encoding='utf-8') as f:
    json.dump(facturas, f, ensure_ascii=False, indent=2)

print(f'Guardado: {OUTPUT_PATH}')

Guardado: ..\data\processed\facturas_validadas.json


In [5]:
import pandas as pd

df = pd.DataFrame(facturas)

# Resumen de validación
print('=== RESUMEN VALIDACIÓN SUNAT ===')
print(df['validacion_ok'].value_counts(dropna=False).rename({True: 'ACTIVO/HABIDO ✅', False: 'Problema ❌', None: 'Indeterminado ⚠️'}))

print('\n=== FACTURAS CON PROBLEMAS ===')
cols = ['tipo_operacion', 'ruc_emisor', 'razon_social_emisor', 'sunat_estado', 'sunat_condicion', 'validacion_nota']
problemas = df[df['validacion_ok'] != True][cols]
if len(problemas):
    print(problemas.to_string(index=False))
else:
    print('  Ninguna — todos los RUCs validados correctamente')

=== RESUMEN VALIDACIÓN SUNAT ===
validacion_ok
Indeterminado ⚠️    3
Name: count, dtype: int64

=== FACTURAS CON PROBLEMAS ===
tipo_operacion  ruc_emisor         razon_social_emisor sunat_estado sunat_condicion       validacion_nota
         VENTA 20608994473 CONTACTO CREATIVO TI S.A.C.         None            None HTTP 401 en intento 2
         VENTA 20608994473 CONTACTO CREATIVO TI S.A.C.         None            None HTTP 401 en intento 2
         VENTA 20608899473 CONTACTO CREATIVO TI S.A.C.         None            None HTTP 401 en intento 2
